In [1]:
# Importaciónes de la biblioteca estándar de Python
import os
import sys
import math
from pathlib import Path
import numpy as np

# Importortaciones de terceros
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import datasets, transforms
from PIL import Image
import matplotlib.pyplot as plt
from compressai.zoo import cheng2020_anchor

#from torchmetrics.image import PeakSignalNoiseRatio, StructuralSimilarityIndexMeasure

from torch.utils.tensorboard import SummaryWriter

import albumentations as A
from albumentations.pytorch import ToTensorV2


In [2]:
# Semilla para reproducibilidad de los experimentos
torch.manual_seed(42)

In [3]:
# Si tenemos disponible GPU, lo usamos
# Chequeamos si tenemos disponible GPU (CUDA)
if torch.cuda.is_available():
    device = "cuda"
# Chequeamos si tenemos disponible aceleración por hardware en un chip de Apple (MPS)
elif torch.backends.mps.is_available():
    device = "mps"
# Por defecto usamos CPU
else:
    device = "cpu"

In [4]:
# Me aseguro de que el directorio raíz del proyecto esté en el sys.path
project_root = Path(os.path.abspath("")).parent

# Añado el directorio raíz al sys.path si no está ya presente
if project_root not in sys.path:
    sys.path.append(str(project_root))

In [5]:
# Importo las funciones de configuración
from src.config import processed_data_dir, models_dir, reports_dir, load_config

from src.utils.datasets import CustomDataset_2_2

#from src.models.convolutional_autoencoder_model.model import ConvolutionalAutoencoder
from src.models.compressai_chang2020_model.train_model import train_model


Current working directory: /home/jorge/development/ImageReconstructionDL/notebooks
Loading configuration from /home/jorge/development/ImageReconstructionDL/src/config.yaml


In [6]:
# Verifico que el dispositivo que se está utilizando
print(f"Usando dispositivo: {device}")

Usando dispositivo: cuda


In [7]:
# Cargo el modelo preentrenado y moverlo al dispositivo
model = cheng2020_anchor(quality=1, pretrained=True).to(device)

In [8]:
nombre_modelo = 'compressai_cheng2020_anchor_2_4_PRESS'
output_model_path = models_dir() / "trained"
os.makedirs(output_model_path, exist_ok=True)

In [9]:
# Cargo la configuración 
config = load_config()

Loading configuration from /home/jorge/development/ImageReconstructionDL/src/config.yaml


In [10]:
# Transformación de las imágenes
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # Asegurar que las imágenes tengan el mismo tamaño
    transforms.ToTensor(),
])

In [11]:
# Definir el pipeline de augmentación
augmentation_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.OneOf([
        A.Rotate(limit=(0, 0), p=0.33),
        A.Rotate(limit=(180, 180), p=0.33),
        A.Compose([A.Rotate(limit=(180, 180), p=1.0), A.HorizontalFlip(p=1.0)], p=0.34)
    ], p=1.0)
])

In [12]:
# Ruta de los datos
# Ruta de los datos
final_data_dir = processed_data_dir() 

In [13]:
# Definir la ruta de los archivos CSV
#train_csv = os.path.join(final_data_dir, 'train.csv')
#val_csv = os.path.join(final_data_dir, 'val.csv')

In [14]:
# Crear el dataset
train_dataset = CustomDataset_2_2(
    csv_file=os.path.join(final_data_dir, 'train_2.csv'),
    transform=transform,
    augmentation_pipeline=augmentation_pipeline,
    use_augmentation=True  # Activar Data Augmentation para el entrenamiento y probar con imágenes aumentadas
)

val_dataset = CustomDataset_2_2(
    csv_file=os.path.join(final_data_dir, 'val_2.csv'),
    transform=transform,
    use_augmentation=False  # No activar Data Augmentation para el conjunto de validación
)

In [15]:
batch_size = 16

In [16]:
# Crear los dataloaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

In [17]:
# Definir el learning rate y la cantidad de épocas
learning_rate = 1e-5
num_epochs = 50

# Defino la función de pérdida y el optimizador
criterion = nn.MSELoss()  # Para la reconstrucción, uso MSELoss
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [18]:
# Scheduler para reducir la tasa de aprendizaje cuando el rendimiento alcanza un plató
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, verbose=True)

/home/jorge/miniconda3/envs/image-recon-dl-env/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "


In [19]:
# Defino directorio base de logs para tensorboard
log_base_dir = reports_dir()/'logs'
os.makedirs(log_base_dir, exist_ok=True)

In [20]:
# Defino subdirectorio de logs para este modelo
log_dir = log_base_dir / nombre_modelo
os.makedirs(log_dir, exist_ok=True)

In [21]:
# Inicializar tensorboard
writer = SummaryWriter(log_dir=str(log_dir), comment="Run_compressai_cheng2020_anchor_2_3")

In [22]:
%load_ext tensorboard

%tensorboard --logdir=../reports/logs/compressai_cheng2020_anchor_2_4_PRESS --host 0.0.0.0 --port 6006

In [23]:
train_losses, val_losses, val_psnr_values, val_ssim_values, compression_ratios = (
    train_model(
        model,
        train_loader,
        val_loader,
        criterion,
        optimizer,
        scheduler,
        output_model_path, 
        nombre_modelo,
        num_epochs=num_epochs,
        device=device,
        writer=writer 
    )
)
# Tardo 98 minutos en entrenar el modelo - GPU NVIDIA GeForce RTX 3080 Ti
# Tardo 23.25 minutos en entrenar el modelo - GPU NVIDIA GeForce RTX 4070 Ti Super

Epoch [1/50], Train Loss: 0.0014, Val Loss: 0.0011, Val PSNR: 30.1333, Val SSIM: 0.8909, Compression Ratio: 140.88
Epoch [2/50], Train Loss: 0.0010, Val Loss: 0.0010, Val PSNR: 31.1205, Val SSIM: 0.9150, Compression Ratio: 100.48
Epoch [3/50], Train Loss: 0.0009, Val Loss: 0.0009, Val PSNR: 31.5698, Val SSIM: 0.9234, Compression Ratio: 77.38
Epoch [4/50], Train Loss: 0.0008, Val Loss: 0.0008, Val PSNR: 31.8744, Val SSIM: 0.9288, Compression Ratio: 61.28
Epoch [5/50], Train Loss: 0.0008, Val Loss: 0.0008, Val PSNR: 32.1151, Val SSIM: 0.9331, Compression Ratio: 55.61
Epoch [6/50], Train Loss: 0.0007, Val Loss: 0.0007, Val PSNR: 32.3436, Val SSIM: 0.9361, Compression Ratio: 51.86


KeyboardInterrupt: 

In [ ]:
# Los elementos dentro de val_psnr_values son tensores de PyTorch, hay que convertir cada uno a un array de NumPy
val_psnr_values = [v.cpu().numpy() if isinstance(v, torch.Tensor) else v for v in val_psnr_values]

# Los elementos dentro de val_ssim_values son tensores de PyTorch, hay que convertir cada uno a un array de NumPy
val_ssim_values = [v.cpu().numpy() if isinstance(v, torch.Tensor) else v for v in val_ssim_values]

# Graficar las pérdidas de entrenamiento y validación
plt.figure(figsize=(10, 5))
plt.plot(train_losses, label='Training Loss')
plt.plot(val_losses, label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.title('Training and Validation Loss')
plt.show()

# Graficar PSNR y SSIM de validación
plt.figure(figsize=(10, 5))
plt.plot(val_psnr_values, label='Validation PSNR')
plt.xlabel('Epochs')
plt.ylabel('PSNR')
plt.legend()
plt.title('Validation PSNR')
plt.show()

plt.figure(figsize=(10, 5))
plt.plot(val_ssim_values, label='Validation SSIM')
plt.xlabel('Epochs')
plt.ylabel('SSIM')
plt.legend()
plt.title('Validation SSIM')
plt.show()

# Graficar Ratio de Compresión
plt.figure(figsize=(10, 5))
plt.plot(compression_ratios, label='Compression Ratio')
plt.xlabel('Epochs')
plt.ylabel('Compression Ratio')
plt.legend()
plt.title('Compression Ratio')
plt.show()